# Домашняя работа — Занятие 26: PyTorch

Формат как мы обсуждали:
- На слайде: базовые идеи (тензор, GPU, autograd, цикл обучения)
- На практике: пайплайн обучения в PyTorch
- Здесь: два режима домашки

## Режим 1: Debug Challenge
В коде специально оставлены ошибки. Твоя задача — найти их, исправить и коротко объяснить, что было не так.

## Режим 2: Less Scaffold
Дано меньше опоры: нужно дописать куски кода (TODO) так, чтобы всё работало и давало адекватное качество.


In [1]:
# Импорты (минимум)
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


## Быстрый прогрев (минимум)

1) Создай тензор `x` формы (2, 3) со случайными значениями  
2) Посчитай `x_mean` по столбцам (dim=0)  
3) Сделай `x_gpu = x.to(device)`  

Оставь здесь свой код и выводы.


In [2]:
# TODO: прогрев
x = torch.rand((2, 3))
x_mean = x.mean(dim=0)
x_gpu = x.to(device)

print("x:\n", x)
print("x_mean:", x_mean)
print("x_gpu.device:", x_gpu.device)

x:
 tensor([[0.8823, 0.9150, 0.3829],
        [0.9593, 0.3904, 0.6009]])
x_mean: tensor([0.9208, 0.6527, 0.4919])
x_gpu.device: cpu


# Режим 1 — Debug Challenge

Ниже пайплайн классификации (Wine). В нём спрятаны ошибки из темы занятия:
- dtype для меток
- перенос на device
- неправильная функция потерь / активация
- накопление градиентов
- eval/no_grad
- DataLoader для теста

Твоя задача:
1) Запусти. Посмотри, где ломается или почему качество странное.  
2) Исправь.  
3) Коротко подпиши рядом (в markdown), какие ошибки ты нашёл.

Подсказка по стилю: как на практике — исправляем по одной проблеме и проверяем снова.


## A) Данные (в этом блоке есть минимум 2 ошибки)

In [3]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_wine()
X = data.data
y = data.target

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_test_np = scaler.transform(X_test_np)

# Ошибка 1: y должен быть torch.long для nn.CrossEntropyLoss
X_train = torch.tensor(X_train_np, dtype=torch.float32)
X_test  = torch.tensor(X_test_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
y_test  = torch.tensor(y_test_np, dtype=torch.long)

train_ds = TensorDataset(X_train, y_train)
test_ds  = TensorDataset(X_test, y_test)

# Ошибка 2: shuffle=False для теста для стабильности оценки
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

print("OK: loaders created")

OK: loaders created


## B) Модель (в этом блоке есть 1 ошибка концепта)

In [4]:
class MLP(nn.Module):
    def __init__(self, in_features: int, num_classes: int):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.out = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        # Удален Softmax, так как CrossEntropyLoss принимает сырые логиты
        x = self.out(x)
        return x

model = MLP(in_features=X_train.shape[1], num_classes=len(np.unique(y))).to(device)
print(model)

MLP(
  (fc1): Linear(in_features=13, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (out): Linear(in_features=32, out_features=3, bias=True)
  (relu): ReLU()
)


## C) Обучение (в этом блоке есть минимум 3 ошибки)

In [5]:
def accuracy_from_logits(logits: torch.Tensor, y_true: torch.Tensor) -> float:
    preds = torch.argmax(logits, dim=1)
    return (preds == y_true).float().mean().item()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 20
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    epoch_acc = 0.0
    n_batches = 0

    for xb, yb in train_loader:
        # Батчи перенесены на device, а yb теперь корректного типа (long)
        xb = xb.to(device)
        yb = yb.to(device)

        # Обнуление градиентов перед шагом
        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += accuracy_from_logits(logits.detach(), yb)
        n_batches += 1

    if epoch % 5 == 0:
        print(f"epoch={epoch:03d} loss={epoch_loss/n_batches:.4f} acc={epoch_acc/n_batches:.4f}")

epoch=005 loss=0.8474 acc=0.9812
epoch=010 loss=0.4368 acc=0.9750
epoch=015 loss=0.1611 acc=0.9875
epoch=020 loss=0.0663 acc=0.9938


## D) Тест (в этом блоке есть минимум 2 ошибки)

In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# Модель переведена в режим оценки
model.eval()

all_preds = []
all_true = []

with torch.no_grad():
    for xb, yb in test_loader:
        # Данные перенесены на device
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        preds = torch.argmax(logits, dim=1)

        all_preds.append(preds.cpu())
        all_true.append(yb.cpu())

all_preds = torch.cat(all_preds).numpy()
all_true = torch.cat(all_true).numpy()

print("Test accuracy:", (all_preds == all_true).mean())
print("\nClassification report:\n", classification_report(all_true, all_preds))
print("Confusion matrix:\n", confusion_matrix(all_true, all_preds))

Test accuracy: 0.9722222222222222

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.93      1.00      0.97        14
           2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

Confusion matrix:
 [[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]


# Режим 2 — Less Scaffold (меньше опоры)

Сделай свой рабочий пайплайн с нуля, но по той же логике, что на практике.

Требование:
- DataLoader для train/test
- MLP через nn.Module
- обучение по «5 шагам»
- отчёт по качеству на тесте

Ниже — заготовки без готовых ответов.


## 1) Данные

In [7]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_wine()
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target
)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np)
X_test_np = scaler.transform(X_test_np)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_np, dtype=torch.float32), torch.tensor(y_train_np, dtype=torch.long)),
    batch_size=32, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test_np, dtype=torch.float32), torch.tensor(y_test_np, dtype=torch.long)),
    batch_size=32, shuffle=False
)

num_features = data.data.shape[1]
num_classes = len(np.unique(data.target))

## 2) Модель

In [8]:
class MLP(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = MLP(num_features, num_classes).to(device)

## 3) Обучение

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 51):
    model.train()
    total_loss, total_acc = 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += (out.argmax(1) == yb).float().mean().item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss={total_loss/len(train_loader):.4f}, Acc={total_acc/len(train_loader):.4f}")

Epoch 10: Loss=0.3972, Acc=0.9732
Epoch 20: Loss=0.0728, Acc=0.9938
Epoch 30: Loss=0.0241, Acc=1.0000
Epoch 40: Loss=0.0123, Acc=1.0000
Epoch 50: Loss=0.0075, Acc=1.0000


## 4) Тест

In [10]:
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)
        all_preds.append(out.argmax(1).cpu())
        all_true.append(yb)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).numpy()

print(f"Final Accuracy: {(y_pred == y_true).mean():.4f}")
print(classification_report(y_true, y_pred))

Final Accuracy: 0.9722
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.93      1.00      0.97        14
           2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



## Что сдаём по этой тетрадке

1) В Debug Challenge: исправленный код + короткие подписи, какие ошибки нашёл  
2) В Less Scaffold: рабочая реализация пайплайна  
3) Короткий вывод: какие 2–3 вещи сильнее всего влияют на качество (lr, batch_size, архитектура)


Выводы:
Сильнее всего влияют на качество:
- Правильный Learning Rate: если он слишком большой, модель будет прыгать мимо решения и не обучится, если слишком маленький,то обучение затянется на очень долго
- Масштабирование данных. Нужно привести всё к единому масштабу, иначе модель будет обращать внимание только на большие цифры